# RAG Ingest Pipeline — Interactive Walkthrough

This notebook walks through every stage of `ingest_reports.py`, showing the
data as it transforms at each step:

| Step | What happens |
|------|--------------|
| 1 Extract | Pull raw text out of a PDF page |
| 2 Clean   | Strip boilerplate headers that repeat on every page |
| 3 Chunk   | Split text into segments sized for retrieval |
| 4 Embed   | Turn each chunk into a vector with OpenRouter |


requires `OPEN_ROUTER_API_KEY`

In [24]:
import sys
!uv pip install --python {sys.executable} numpy pandas python-dotenv pypdf openai supabase


Using Python 3.13.12 environment at: /home/bart/repos/agentic-rag/part_1/.venv
Checked 6 packages in 2ms


In [42]:
import os, getpass
os.environ["OPEN_ROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

In [34]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from pypdf import PdfReader
from openai import OpenAI
from supabase import create_client

load_dotenv()

# ── Configuration ────────────────────────────────────────────────────────────
CHUNK_SIZE  = 500   # soft max characters per chunk
EMBED_MODEL = "openai/text-embedding-3-small"  # via OpenRouter
EMBED_BATCH = 50
DATASET     = "northwind_reports"
REPORTS_DIR = Path("./reports")

# Boilerplate lines printed at the top of every PDF page
_BOILERPLATE = (
    re.compile(r"Northwind Traders\s+·"),  # company + doc-type banner
    re.compile(r"^Page \d+$"),             # standalone page-number line
)

# ── Helper functions (identical to ingest_reports.py) ────────────────────────
def clean_page_text(text: str) -> str:
    """Remove boilerplate lines that appear at the top of every PDF page."""
    lines = text.strip().splitlines()
    return "\n".join(
        line for line in lines
        if not any(pat.search(line.strip()) for pat in _BOILERPLATE)
    ).strip()

def split_text(text: str) -> list[str]:
    """Group text into chunks up to CHUNK_SIZE characters."""
    # Split on blank lines where possible; fall back to per-line if pypdf omits them
    units = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if len(units) == 1:
        units = [line.strip() for line in text.splitlines() if line.strip()]
    chunks, current = [], ""
    for unit in units:
        candidate = (current + "\n" + unit).strip() if current else unit
        if current and len(candidate) > CHUNK_SIZE:
            chunks.append(current)
            current = unit
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks

print(f"Reports directory: {REPORTS_DIR.resolve()}")
print(f"PDFs found: {len(list(REPORTS_DIR.glob('*.pdf')))}")

Reports directory: /home/bart/repos/agentic-rag/part_3/explanation_materials/reports
PDFs found: 10


---
## Stage 1 — Extract raw text from a PDF

`pypdf` reads the PDF and extracts text page by page.
Let's pick one report and look at what page 1 actually contains.

In [35]:
pdfs = sorted(REPORTS_DIR.glob("*.pdf"))
for i, p in enumerate(pdfs):
    print(f"  [{i}] {p.name}")

  [0] 2025_annual_report.pdf
  [1] americas_market_report_2025.pdf
  [2] customer_accounts_review_2025.pdf
  [3] european_market_intelligence.pdf
  [4] inventory_reorder_alert_q4_2025.pdf
  [5] product_portfolio_analysis_2025.pdf
  [6] q4_2025_executive_summary.pdf
  [7] sales_team_review_h2_2025.pdf
  [8] shipper_performance_report.pdf
  [9] supplier_review_2025.pdf


In [36]:
# Pick any report by index
sample_pdf = pdfs[0]
reader = PdfReader(str(sample_pdf))

raw_page1 = reader.pages[0].extract_text() or ""

print(f"File : {sample_pdf.name}")
print(f"Pages: {len(reader.pages)}")
print(f"\n── Raw text from page 1 ({'─' * 40})")
print(raw_page1)

File : 2025_annual_report.pdf
Pages: 3

── Raw text from page 1 (────────────────────────────────────────)
Northwind Traders  ·  Internal Document
Annual Report 2025
Page 1
NORTHWIND TRADERS
Annual Report 2025
Fiscal Year January – December 2025
EXECUTIVE SUMMARY
Northwind Traders processed 408 customer orders in fiscal year 2025, spanning 21 countries and generating
total net revenue of $660,788. Beverages was the highest-revenue product category at $142,600 (21.6% of
total), followed by Meat/Poultry at $138,490 (21.0%) and Dairy Products at $125,580 (19.0%). The Seafood
category experienced a mid-year supply disruption (Q2 2025) caused by late deliveries from Pacific Rim
partners Tokyo Traders and Lyngbysild; the disruption reduced estimated Seafood revenue by $18,400 and
was resolved in July 2025 through revised supplier SLAs and the onboarding of New England Seafood
Cannery as a contingency supplier.
The United States remained Northwind Traders' largest single market in 2025, contr

---
## Stage 2 — Clean the boilerplate

Every page starts with three lines that carry no useful information:
```
Northwind Traders  ·  Internal Document
<document title>
Page N
```
If we leave these in, every chunk will contain them — wasting token space
and polluting the embedding with noise. `clean_page_text` strips them.

In [37]:
cleaned_page1 = clean_page_text(raw_page1)

removed_lines = set(raw_page1.splitlines()) - set(cleaned_page1.splitlines())
print("Lines removed:")
for line in removed_lines:
    print(f"  ✕  {repr(line)}")

print(f"\n── Cleaned text ({'─' * 44})")
print(cleaned_page1)

Lines removed:
  ✕  'Page 1'
  ✕  'Northwind Traders  ·  Internal Document'

── Cleaned text (────────────────────────────────────────────)
Annual Report 2025
NORTHWIND TRADERS
Annual Report 2025
Fiscal Year January – December 2025
EXECUTIVE SUMMARY
Northwind Traders processed 408 customer orders in fiscal year 2025, spanning 21 countries and generating
total net revenue of $660,788. Beverages was the highest-revenue product category at $142,600 (21.6% of
total), followed by Meat/Poultry at $138,490 (21.0%) and Dairy Products at $125,580 (19.0%). The Seafood
category experienced a mid-year supply disruption (Q2 2025) caused by late deliveries from Pacific Rim
partners Tokyo Traders and Lyngbysild; the disruption reduced estimated Seafood revenue by $18,400 and
was resolved in July 2025 through revised supplier SLAs and the onboarding of New England Seafood
Cannery as a contingency supplier.
The United States remained Northwind Traders' largest single market in 2025, contributing $245,2

---
## Stage 3 — Chunk the text

Embedding models have a token limit, and short focused chunks retrieve
better than huge pages. Our strategy:

- Split on **blank lines** (paragraph/section breaks) where possible
- When pypdf strips blank lines (common), fall back to **line-by-line grouping**
- Accumulate lines until the next one would push us past `CHUNK_SIZE`

This keeps table rows and sentences intact — a naive character-slice would
cut right through a table row mid-value.

In [38]:
# Chunk every page of the sample PDF
all_chunks = []
for page_num, page in enumerate(reader.pages, start=1):
    text = clean_page_text(page.extract_text() or "")
    for idx, chunk in enumerate(split_text(text)):
        all_chunks.append({"page": page_num, "chunk_index": idx, "chars": len(chunk), "content": chunk})

df = pd.DataFrame(all_chunks)
print(f"Total chunks: {len(df)}  |  avg size: {df['chars'].mean():.0f} chars  |  max: {df['chars'].max()} chars")
print()
df[["page", "chunk_index", "chars", "content"]].style.set_properties(
    **{"white-space": "pre-wrap", "text-align": "left", "max-width": "700px"}
)

Total chunks: 12  |  avg size: 358 chars  |  max: 500 chars



,page,chunk_index,chars,content
0,1,0,424,"Annual Report 2025 NORTHWIND TRADERS Annual Report 2025 Fiscal Year January – December 2025 EXECUTIVE SUMMARY Northwind Traders processed 408 customer orders in fiscal year 2025, spanning 21 countries and generating total net revenue of $660,788. Beverages was the highest-revenue product category at $142,600 (21.6% of total), followed by Meat/Poultry at $138,490 (21.0%) and Dairy Products at $125,580 (19.0%). The Seafood"
1,1,1,447,"category experienced a mid-year supply disruption (Q2 2025) caused by late deliveries from Pacific Rim partners Tokyo Traders and Lyngbysild; the disruption reduced estimated Seafood revenue by $18,400 and was resolved in July 2025 through revised supplier SLAs and the onboarding of New England Seafood Cannery as a contingency supplier. The United States remained Northwind Traders' largest single market in 2025, contributing $245,220 (37.1% of"
2,1,2,500,"total revenue) from 13 active customer accounts. Germany was the largest European market at $118,560 (17.9%), driven by anchor accounts QUICK-Stop (Cunewalde) and ERNST HANDEL (Graz). Austria contributed $89,410 (13.5%) primarily from BOTTM-Fuest. KEY PERFORMANCE METRICS — FISCAL YEAR 2025 Metric FY 2025 H2 2024 Change Total orders 408 152 +168% Net revenue $660,788 $243,088 +172% Average order value $1,619 $1,599 +1.3% Active customers 89 71 +25% Countries served 21 19 +2 Products catalogued 77"
3,1,3,464,"77 — Products discontinued 8 8 — REVENUE BY PRODUCT CATEGORY — 2025 Category Revenue Share Orders Key observation Beverages $142,600 21.6% 128 Premium SKUs (Côte de Blaye) drive value Meat/Poultry $138,490 21.0% 112 Strong restaurant-trade demand Dairy Products $125,580 19.0% 109 Cheese lines grew 18% by volume Confections $82,660 12.5% 87 Q4 2025 holiday spike of +34% Seafood $64,420 9.8% 74 H1 supply disruption; full recovery in H2 Condiments $46,250 7.0% 68"
4,1,4,37,Margin pressure from EU private-label
5,2,0,500,"Annual Report 2025 Category Revenue Share Orders Key observation Produce $33,820 5.1% 41 Seasonal variation; stable accounts Grains/Cereals $26,968 4.1% 38 Steady institutional buyer demand Total $660,788 100% 657 REVENUE BY GEOGRAPHY — TOP 6 MARKETS — 2025 Country Revenue Share Customers Top account USA $245,220 37.1% 13 Save-a-Lot Markets (Boise, ID) Germany $118,560 17.9% 11 QUICK-Stop (Cunewalde) Austria $89,410 13.5% 2 BOTTM-Fuest Brazil $66,790 10.1% 9 Hanari Carnes (Rio de Janeiro) France"
6,2,1,103,"$44,920 6.8% 11 Bon app' (Marseille) Other $95,888 14.5% 43 21 further countries Total $660,788 100% 89"
7,3,0,458,"Annual Report 2025 OPERATIONAL HIGHLIGHTS Seafood supply disruption (Q2 2025). Between April and June 2025, Northwind Traders experienced delivery delays from Tokyo Traders (Tokyo, Japan) and Lyngbysild (Lyngby, Denmark), affecting the Konbu, Tofu, and Rogede sild product lines. The combined revenue impact was estimated at $18,400. Corrective actions included revised contractual SLAs with Tokyo Traders, a 30-day advance shipment notification requirement,"
8,3,1,412,"and the qualification of New England Seafood Cannery (Boston, USA) as a contingency Seafood supplier from July 2025. Condiments margin pressure. Northwind Traders' Condiments gross margin declined from 38% to 31% in H1 2025, driven by competition from European private-label alternatives. A pricing review is scheduled with key Condiments accounts in January 2026 to defend volume without further margin erosion."
9,3,2,437,"Logistics performance. United Package handled 163 orders (40% of total volume) at an average freight cost of $78.40 per order, achieving 94.2% on-time delivery. Speedy Express carried 91 orders (22%) at $104.60 average freight with the highest on-time rate of 97.1%. Federal Shipping handled 154 orders (38%), primarily bulk and pallet-level consignments, at $86.20 average freight and 91.1% on-time performance. STRATEGIC OUTLOOK — 2026"


In [39]:
# Inspect any individual chunk
CHUNK_TO_INSPECT = 0

row = df.iloc[CHUNK_TO_INSPECT]
print(f"Chunk {CHUNK_TO_INSPECT}  (page {row.page}, {row.chars} chars)")
print("─" * 60)
print(row.content)

Chunk 0  (page 1, 424 chars)
────────────────────────────────────────────────────────────
Annual Report 2025
NORTHWIND TRADERS
Annual Report 2025
Fiscal Year January – December 2025
EXECUTIVE SUMMARY
Northwind Traders processed 408 customer orders in fiscal year 2025, spanning 21 countries and generating
total net revenue of $660,788. Beverages was the highest-revenue product category at $142,600 (21.6% of
total), followed by Meat/Poultry at $138,490 (21.0%) and Dairy Products at $125,580 (19.0%). The Seafood


---
## Stage 4 — Create embeddings

An **embedding** is a list of numbers that captures the *meaning* of a piece
of text. Chunks that mean similar things end up close together in vector space.

We use **`openai/text-embedding-3-small`** via OpenRouter:
- 1 536-dimensional vectors
- Same OpenAI SDK — just a different `base_url`
- Billed per token through your OpenRouter account

In [43]:
# OpenAI client pointed at OpenRouter
openai_client = OpenAI(
    api_key=os.getenv("OPEN_ROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

print(f"Model : {EMBED_MODEL}")
print("Client ready.")

Model : openai/text-embedding-3-small
Client ready.


In [44]:
# Embed the first few chunks to show what a vector looks like
sample_texts = df["content"].tolist()[:4]

response = openai_client.embeddings.create(model=EMBED_MODEL, input=sample_texts)
sample_vectors = [item.embedding for item in response.data]

print(f"Vector dimensionality : {len(sample_vectors[0])}")
print(f"Embeddings returned   : {len(sample_vectors)}")
print()
for i, vec in enumerate(sample_vectors):
    preview = ", ".join(f"{v:.4f}" for v in vec[:6])
    print(f"Chunk {i}: [{preview}, ...]")

Vector dimensionality : 1536
Embeddings returned   : 4

Chunk 0: [-0.0065, 0.0071, 0.0446, 0.0022, 0.0081, -0.0153, ...]
Chunk 1: [-0.0212, 0.0267, 0.0797, 0.0279, 0.0406, 0.0033, ...]
Chunk 2: [-0.0463, 0.0126, 0.0125, -0.0144, -0.0026, -0.0043, ...]
Chunk 3: [0.0224, -0.0153, -0.0091, -0.0115, 0.0079, -0.0012, ...]
